In [321]:
import pandas as pd
import numpy as np

In [322]:
# Load dataset
df = pd.read_csv("../data/Metro_Manila_Traffic_Incidents_2025.csv")


In [323]:
#Size
df.shape

(1010, 17)

In [324]:
#Data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Incident_ID        1010 non-null   str    
 1   Date               1010 non-null   str    
 2   Time               1010 non-null   str    
 3   City               1010 non-null   str    
 4   Road_Name          1010 non-null   str    
 5   Vehicle_Type       1010 non-null   str    
 6   Accident_Type      1010 non-null   str    
 7   Severity           605 non-null    str    
 8   Weather_Condition  1010 non-null   str    
 9   Road_Condition     614 non-null    str    
 10  Cause              746 non-null    str    
 11  Driver_Age         996 non-null    float64
 12  Driver_Gender      1010 non-null   str    
 13  Injury_Count       845 non-null    float64
 14  Damage_Cost_PHP    1002 non-null   float64
 15  Latitude           1010 non-null   float64
 16  Longitude          1010 non-null   

In [325]:
#Columns names
df.columns

Index(['Incident_ID', 'Date', 'Time', 'City', 'Road_Name', 'Vehicle_Type',
       'Accident_Type', 'Severity', 'Weather_Condition', 'Road_Condition',
       'Cause', 'Driver_Age', 'Driver_Gender', 'Injury_Count',
       'Damage_Cost_PHP', 'Latitude', 'Longitude'],
      dtype='str')

In [326]:
#Clean column names using snake_case and in lowercase

df.columns = df.columns.str.lower()
df.columns

Index(['incident_id', 'date', 'time', 'city', 'road_name', 'vehicle_type',
       'accident_type', 'severity', 'weather_condition', 'road_condition',
       'cause', 'driver_age', 'driver_gender', 'injury_count',
       'damage_cost_php', 'latitude', 'longitude'],
      dtype='str')

In [327]:
# Removes rows where every column value is identical to another row.
df = df.drop_duplicates()

#Verify
df.duplicated().sum()

np.int64(0)

In [359]:
#Strip whitespace
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include=["object", "string"]):
    df[col] = df[col].str.strip()

#Verify
for col in df.select_dtypes(include=["object", "string"]):
    print(col, df[col].str.contains(r"^\s|\s$", regex=True).sum())




incident_id 0
city 0
road_name 0
vehicle_type 0
accident_type 0
severity 0
weather_condition 0
road_condition 0
cause 0
driver_gender 0


In [329]:
# Validate primary key integrity:
# Ensure incident_id has no duplicates and matches total row count
total_rows = df.shape[0]
unique_ids = df["incident_id"].nunique()

print(total_rows, unique_ids)


1000 1000


In [330]:
# Datetime Standardization & Validation

# Combine date and time columns into a single datetime column
# Invalid or improperly formatted values will be converted to NaT
df["datetime"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    errors="coerce"
)
invalid = df[df["datetime"].isna()]

#Verify
df["datetime"].isna().sum()

# Check Min and Max(All should be in range of year 2025)
df["datetime"].min()
df["datetime"].max()

#Drop date and time column
df = df.drop(columns=["date", "time"])
df.columns




Index(['incident_id', 'city', 'road_name', 'vehicle_type', 'accident_type',
       'severity', 'weather_condition', 'road_condition', 'cause',
       'driver_age', 'driver_gender', 'injury_count', 'damage_cost_php',
       'latitude', 'longitude', 'datetime'],
      dtype='str')

In [331]:
#Standardize City names
df["city"] = df["city"].replace({
    "manila": "Manila",
    "QUEZON CITY": 'Quezon City',
})

#Verify
df["city"].value_counts(dropna=False)

city
Pasig          112
Quezon City    107
Manila         100
Makati          93
Las Piñas       59
Navotas         58
Parañaque       52
Malabon         52
Marikina        51
Taguig          49
Valenzuela      48
Pasay           47
Caloocan        46
Mandaluyong     45
San Juan        42
Pateros         39
Name: count, dtype: int64

In [332]:
# Analyze frequency of road_name values to identify inconsistencies and dominant entries
df["road_name"].value_counts(dropna=False)

road_name
Ortigas Ave         111
Roxas Blvd          108
Quezon Ave          105
Taft Ave            105
Commonwealth Ave    103
C5                  101
Aurora Blvd          95
España Blvd          94
EDSA                 91
Katipunan Ave        87
Name: count, dtype: int64

In [333]:
# Analyze frequency of vehicle_type values to identify inconsistencies and dominant entries
df["vehicle_type"].value_counts(dropna=False)

vehicle_type
Truck         171
Taxi          158
UV Express    151
Bus           140
Car           134
Motorcycle    123
Jeepney       123
Name: count, dtype: int64

In [334]:
# Analyze frequency of accident_type values to identify inconsistencies and dominant entries
df["accident_type"].value_counts(dropna=False)

accident_type
Hit and Run    217
Rear-end       208
Collision      193
Side-swipe     192
Pedestrian     190
Name: count, dtype: int64

In [335]:
# Analyze frequency of severity values to identify inconsistencies and dominant entries
df["severity"].value_counts(dropna=False)

severity
NaN      401
Minor    211
Major    202
Fatal    186
Name: count, dtype: int64

In [336]:
# Analyze frequency of weather_condition values to identify inconsistencies and dominant entries
df["weather_condition"].value_counts(dropna=False)

#Standardize City names
df["weather_condition"] = df["weather_condition"].replace({
    "RAIN": "Rain",
    "clear": 'Clear',
})

#Verify
df["weather_condition"].value_counts(dropna=False)

weather_condition
Clear     344
Rain      324
Cloudy    180
Storm     152
Name: count, dtype: int64

In [337]:
# Analyze frequency of road_condition values to identify inconsistencies and dominant entries
df["road_condition"].value_counts(dropna=False)

road_condition
NaN         391
Dry         222
Wet         197
Slippery    190
Name: count, dtype: int64

In [338]:
# Analyze frequency of cause values to identify inconsistencies and dominant entries
df["cause"].value_counts(dropna=False)

cause
NaN                   262
Distracted Driving    144
Drunk Driving         130
Mechanical Failure    127
Reckless Driving      126
Traffic Violation     108
Overspeeding          103
Name: count, dtype: int64

In [339]:
# Data Quality Check: Validate and standardize driver_age
#Review distribution and summary statistics
df["driver_age"].describe()

#Check for missing values
df["driver_age"].isna().sum()

#Confirm current data type
df["driver_age"].dtype

#Identify unexpected decimal values (age should be whole number)
df[df["driver_age"] % 1 != 0]

#Detect unrealistic age values (domain validation)
df[df["driver_age"] < 0]
df[df["driver_age"] > 110]

#Convert to nullable integer type after validation
df["driver_age"] = df["driver_age"].astype("Int64")

#Verify: 
df["driver_age"].dtype

Int64Dtype()

In [340]:
# Analyze frequency of driver_gender values to identify inconsistencies and dominant entries
df["driver_gender"].value_counts(dropna=False)

#Standardize Gender
df["driver_gender"] = df["driver_gender"].replace({
    "male": "Male",
    "FEMALE": 'Female',
})

#Verify
df["driver_gender"].value_counts(dropna=False)

driver_gender
Female    501
Male      499
Name: count, dtype: int64

In [341]:
# Analyze frequency of injury_count values to identify inconsistencies and dominant entries
df["injury_count"].value_counts(dropna=False)

df["injury_count"].dtype

dtype('float64')

In [342]:
#Remove negative damage costs

#Check if Any Negative Numbers Exist
(df["damage_cost_php"] < 0).any()

#Check How Many Negative Values
(df["damage_cost_php"] < 0).sum()

#See those negative rows
df[df["damage_cost_php"] < 0]

#Replacte negative damage cost by NaN
df.loc[df["damage_cost_php"]< 0, "damage_cost_php"] = np.nan

#Verify
(df["damage_cost_php"] < 0).sum()

np.int64(0)

In [358]:
#Data Quality Check: Validate Geographic Coordinates

#Ensure numeric type (convert if necessary)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

#Check for missing values
lat_missing = df["latitude"].isna().sum()
lon_missing = df["longitude"].isna().sum()

# Latitude must be between -90 and 90
invalid_lat_global = df[
    (df["latitude"] < -90) | (df["latitude"] > 90)
]

# Longitude must be between -180 and 180
invalid_lon_global = df[
    (df["longitude"] < -180) | (df["longitude"] > 180)
]

# Confirm values fall within Philippines geographic range
# Approximate PH bounds: Latitude (4–21), Longitude (116–127)
invalid_lat_ph = df[
    (df["latitude"] < 4) | (df["latitude"] > 21)
]

invalid_lon_ph = df[
    (df["longitude"] < 116) | (df["longitude"] > 127)
]

# Detect potential swapped coordinates
# (Latitude unusually high for PH or longitude unusually low)
potential_swapped = df[
    (df["latitude"] > 90) | (df["longitude"] < 0)
]

#Detect placeholder coordinates (0,0)
zero_coordinates = df[
    (df["latitude"] == 0) & (df["longitude"] == 0)
]

#Verify
print("Missing Latitude:", lat_missing)
print("Missing Longitude:", lon_missing)
print("Invalid Global Latitude:", len(invalid_lat_global))
print("Invalid Global Longitude:", len(invalid_lon_global))
print("Invalid PH Latitude:", len(invalid_lat_ph))
print("Invalid PH Longitude:", len(invalid_lon_ph))
print("Potential Swapped:", len(potential_swapped))
print("Zero Coordinates:", len(zero_coordinates))


Missing Latitude: 0
Missing Longitude: 0
Invalid Global Latitude: 0
Invalid Global Longitude: 0
Invalid PH Latitude: 0
Invalid PH Longitude: 0
Potential Swapped: 0
Zero Coordinates: 0
